# Day 50 — Capstone Model Optimization

This notebook evaluates the Day 49 baseline, compares multiple algorithms, tunes hyperparameters with cross-validation, and examines train/test generalization.

In [ ]:
from pathlib import Path
import sys, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.preprocessing import engineer_features, build_preprocessor
df = engineer_features(pd.read_csv(ROOT/'data/raw/customer_data.csv'))
X = df.drop(columns=['churn','customer_id']); y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
preprocessor = build_preprocessor(X_train)
df.head()

## 1. Baseline and candidate models

The comparison uses Logistic Regression as the interpretable baseline, Random Forest for nonlinear interactions, and Gradient Boosting for sequential error correction.

In [ ]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
candidates = {
 'Logistic Regression': (LogisticRegression(max_iter=1500,class_weight='balanced'), {'classifier__C':[.1,1,10],'classifier__solver':['liblinear','lbfgs']}),
 'Random Forest': (RandomForestClassifier(random_state=42,class_weight='balanced',n_jobs=-1), {'classifier__n_estimators':[100,200],'classifier__max_depth':[None,5,10],'classifier__min_samples_leaf':[1,2]}),
 'Gradient Boosting': (GradientBoostingClassifier(random_state=42), {'classifier__n_estimators':[75,125],'classifier__learning_rate':[.03,.1],'classifier__max_depth':[2,3]})
}
comparison=[]
for name,(estimator,params) in candidates.items():
    pipe=Pipeline([('preprocessor',preprocessor),('classifier',estimator)])
    search=GridSearchCV(pipe,params,scoring='roc_auc',cv=cv,n_jobs=-1,refit=True)
    search.fit(X_train,y_train)
    pred=search.predict(X_test); prob=search.predict_proba(X_test)[:,1]
    train_prob=search.predict_proba(X_train)[:,1]
    comparison.append({'model':name,'cv_best_roc_auc':search.best_score_,'test_accuracy':accuracy_score(y_test,pred),'test_precision':precision_score(y_test,pred,zero_division=0),'test_recall':recall_score(y_test,pred,zero_division=0),'test_f1':f1_score(y_test,pred,zero_division=0),'test_roc_auc':roc_auc_score(y_test,prob),'train_roc_auc':roc_auc_score(y_train,train_prob),'generalization_gap':roc_auc_score(y_train,train_prob)-roc_auc_score(y_test,prob),'best_params':search.best_params_})
pd.DataFrame(comparison).sort_values('test_roc_auc',ascending=False)

## 2. Overfitting / underfitting analysis

Compare train and test ROC-AUC. A large train-test gap is a warning sign for overfitting; low scores on both sides can indicate underfitting or weak features.

In [ ]:
results = pd.DataFrame(comparison)
results[['model','train_roc_auc','test_roc_auc','generalization_gap']].sort_values('generalization_gap',ascending=False)

## 3. Engineering tradeoffs

- Logistic Regression: highly interpretable and fast, but limited nonlinear capacity.
- Random Forest: captures nonlinear interactions and is robust, but can become complex and memory-heavy.
- Gradient Boosting: powerful for structured data, but requires careful tuning and can overfit.
- GridSearchCV: improves selection quality at the cost of additional compute.
- Cross-validation: provides a more stable estimate than a single validation split.

## 4. Next steps

Use larger real-world data, temporal validation, probability calibration, threshold optimization based on retention costs, model explainability, and drift monitoring before production deployment.